# Align Bilous He


## Notebook Flow


## 1. Locate the project and data


In [ ]:
from pathlib import Path as _Path
import os as _os
import sys as _sys
_candidates = (_Path.cwd(), *_Path.cwd().parents,
               _Path(globals().get("_code_root", _Path.cwd())))
_code_root = next(p for p in _candidates
                  if (p / 'xenium_pipeline/notebook_setup.py').is_file())
if str(_code_root) not in _sys.path:
    _sys.path.insert(0, str(_code_root))
from xenium_pipeline.notebook_setup import setup_notebook
_code_root, _project_root = setup_notebook()


In [ ]:
from pathlib import Path
import sys

SAMPLE_NAMES = None  # None uses every mapped sample before EXCLUDED_SAMPLES is applied.
EXCLUDED_SAMPLES = [
    'B1_1_GSM9509172',
    'B2_GSM9509160',
    'B3_GSM9509170',
    'B1_2_GSM9509173',
    'B4_GSM9509165',
]
DATA_ROOT_OVERRIDE = None

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (START_DIR, *START_DIR.parents)
        if (candidate / 'dataset_manifest.json').is_file()
        and (candidate / 'Bilous_2026').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the Xenium project root from the current working directory.')

root = Path(DATA_ROOT_OVERRIDE).expanduser().resolve() if DATA_ROOT_OVERRIDE else PROJECT_ROOT / 'Bilous_2026'
print('Project root:', PROJECT_ROOT)
print('Bilous root:', root)


## 2. Check dependencies


In [ ]:
import importlib.util
import subprocess
from importlib.metadata import PackageNotFoundError, version

INSTALL_PACKAGES = False
REQUIREMENTS_PATH = _code_root / 'requirements.txt'
if not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError(REQUIREMENTS_PATH)

REQUIRED_MODULES = {
    'matplotlib': 'matplotlib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'Pillow': 'PIL',
    'pyarrow': 'pyarrow',
    'tifffile': 'tifffile',
    'tqdm': 'tqdm',
    'zarr': 'zarr',
    'imagecodecs': 'imagecodecs',
    'ipywidgets': 'ipywidgets',
}

if INSTALL_PACKAGES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS_PATH)])

missing_modules = [package for package, module in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing_modules:
    raise ModuleNotFoundError(f'Missing modules: {missing_modules}. Set INSTALL_PACKAGES = True and rerun this cell.')

print(f'Python {sys.version.split()[0]}')
print(f'Requirements: {REQUIREMENTS_PATH}')
for package in REQUIRED_MODULES:
    try:
        print(f'{package:12s} {version(package)}')
    except PackageNotFoundError:
        print(f'{package:12s} installed')


## 3. Load the Bilous prepare helper and discover samples


In [ ]:
import importlib

import pandas as pd
from IPython.display import Image as DisplayImage, clear_output, display
import ipywidgets as widgets

# Bilous-specific prepare-stage helper; full tile creation uses xenium_pipeline.standard_xenium.
from xenium_pipeline import bilous_tiles as tile_pipeline
tile_pipeline = importlib.reload(tile_pipeline)
DATASET_CONFIG = tile_pipeline.DATASET_CONFIG

TRANSFORM_DIR = DATASET_CONFIG.transform_path(PROJECT_ROOT)
OUTPUT_DIR = DATASET_CONFIG.output_path(PROJECT_ROOT)

mapping_table = tile_pipeline.read_mapping_table(root)
if SAMPLE_NAMES is None:
    sample_names = mapping_table['sample_folder'].astype(str).tolist()
else:
    sample_names = list(SAMPLE_NAMES)

excluded_samples = list(EXCLUDED_SAMPLES or [])
unknown_excluded = sorted(set(excluded_samples) - set(mapping_table['sample_folder'].astype(str)))
if unknown_excluded:
    print('Excluded samples not found in mapping table:', unknown_excluded)

samples_for_tile_manifest = [sample for sample in sample_names if sample not in set(excluded_samples)]
if not samples_for_tile_manifest:
    raise ValueError('No samples remain for tile manifest after applying EXCLUDED_SAMPLES.')

print('Dataset config:', DATASET_CONFIG.display_name)
print('Transform directory:', TRANSFORM_DIR)
print('Output directory:', OUTPUT_DIR)
print(f'Discovered {len(sample_names)} mapped sample(s):')
for name in sample_names:
    print(f'  - {name}')
print(f'Excluded from tile manifest: {len(set(excluded_samples) & set(sample_names))}')
for name in excluded_samples:
    if name in sample_names:
        print(f'  - {name}')
print(f'Samples for tile manifest: {len(samples_for_tile_manifest)}')
for name in samples_for_tile_manifest:
    print(f'  - {name}')


### Inspect H&E and alignment sources


In [ ]:
input_summary = tile_pipeline.validate_mapping_inputs(
    root,
    sample_names=sample_names,
    transform_dir=TRANSFORM_DIR,
)

he_rows = []
for record in input_summary.to_dict('records'):
    sample = record['sample']
    alignment_csv = TRANSFORM_DIR / f'{sample}_he_imagealignment.csv'
    manual_json = TRANSFORM_DIR / f'{sample}_manual_alignment.json'
    if alignment_csv.is_file():
        alignment_state = 'alignment_csv'
    elif manual_json.is_file():
        alignment_state = 'manual_json_needs_or_has_csv_export'
    else:
        alignment_state = 'missing_alignment'
    he_rows.append({
        'sample': sample,
        'H&E image': record.get('status') == 'valid' and bool(record.get('he_path')),
        'alignment_state': alignment_state,
        'alignment_csv': str(alignment_csv) if alignment_csv.is_file() else '',
        'manual_alignment_json': str(manual_json) if manual_json.is_file() else '',
        'H&E path': record.get('he_path', ''),
        'Xenium cells': record.get('cells_path', ''),
        'input_status': record.get('status', ''),
        'error': record.get('error', ''),
    })
he_table = pd.DataFrame(he_rows)
display(he_table)


## 4. Validate inputs before alignment QC


In [ ]:
validation = tile_pipeline.validate_mapping_inputs(
    root,
    sample_names=sample_names,
    transform_dir=TRANSFORM_DIR,
)
display(validation)

invalid = validation.loc[validation['status'] != 'valid']
if not invalid.empty:
    details = invalid[['sample', 'error']].to_string(index=False)
    raise RuntimeError(f'Fix invalid non-alignment inputs before manual alignment/QC:\n{details}')

print('All selected samples passed non-alignment input validation.')


## 5. Align one sample at a time with pre-cropping tile QC


In [ ]:
from xenium_pipeline import bilous_alignment as manual_landmark_alignment
manual_landmark_alignment = importlib.reload(manual_landmark_alignment)

samples_needing_manual_alignment = manual_landmark_alignment.samples_missing_alignment(
    root,
    output_dir=TRANSFORM_DIR,
    sample_names=sample_names,
)
print('Samples missing `_he_imagealignment.csv`:')
for name in samples_needing_manual_alignment:
    print(f'  - {name}')
if not samples_needing_manual_alignment:
    print('  (none)')


### 5a. Collect paired landmarks for the current sample


In [ ]:
SHOW_LANDMARK_SELECTOR = True
N_LANDMARKS = 15
MAX_DIMENSION = 12000
USE_MORPHOLOGY_MIP = True
CURRENT_ALIGNMENT_SAMPLE_OVERRIDE = "B13_GSM9509161"  # set to a sample name to retry or inspect it again

CURRENT_ALIGNMENT_SAMPLE = (
    CURRENT_ALIGNMENT_SAMPLE_OVERRIDE
    or next(iter(samples_needing_manual_alignment), None)
)

if SHOW_LANDMARK_SELECTOR:
    if CURRENT_ALIGNMENT_SAMPLE is None:
        raise ValueError('No sample is selected for manual alignment.')
    manual_landmark_alignment.show_landmark_selector(
        root,
        CURRENT_ALIGNMENT_SAMPLE,
        n_points=N_LANDMARKS,
        max_dimension=MAX_DIMENSION,
        use_morphology_mip=USE_MORPHOLOGY_MIP,
    )
else:
    print('Set SHOW_LANDMARK_SELECTOR=True to display the current sample.')
    print('CURRENT_ALIGNMENT_SAMPLE =', CURRENT_ALIGNMENT_SAMPLE)


### 5b. Save landmarks and review 9 pre-cropping tile examples


In [ ]:
SAVE_LANDMARKS_AND_PREVIEW = True
LANDMARK_JSON = r'''
{
  "sample_folder": "B13_GSM9509161",
  "sample_name": "B13_GSM9509161",
  "max_dimension": 12000,
  "source_coordinate_space": "native_xenium_preview",
  "source_reference_kind": "native_xenium_morphology_with_cell_nucleus_overlays",
  "source_reference_label": "Native Xenium morphology + cell/nucleus overlays",
  "target_coordinate_space": "he_preview_original",
  "source_points": [
    [
      8655.456143465912,
      2630.634036539813
    ],
    [
      8160.476222493531,
      2260.012245511212
    ],
    [
      8270.550033843223,
      2272.2994114313033
    ],
    [
      7523.126704138844,
      1557.246117491834
    ],
    [
      7083.49367647997,
      1607.1864218783273
    ],
    [
      6287.313918476818,
      2664.0032244907748
    ],
    [
      1746.5701602408626,
      4932.598109868261
    ],
    [
      2241.360809121042,
      5254.454869160833
    ],
    [
      2958.7885245530174,
      6446.679706119445
    ],
    [
      3519.059462607201,
      6278.790658353661
    ],
    [
      3949.0676030073437,
      5849.484133960243
    ],
    [
      7009.610354845328,
      7553.120804827759
    ],
    [
      6422.584829097754,
      8240.16781210196
    ],
    [
      5119.723468502397,
      8838.317711406422
    ],
    [
      5507.689649597713,
      9700.773756706558
    ]
  ],
  "target_points": [
    [
      8623.45933857414,
      8797.581911426389
    ],
    [
      9010.185547404446,
      8280.157977285573
    ],
    [
      8998.221329459204,
      8396.787866089255
    ],
    [
      9754.929681332527,
      7610.9759321546635
    ],
    [
      9697.81144553014,
      7153.31109005224
    ],
    [
      8592.718438775082,
      6321.17066222033
    ],
    [
      6220.448573450895,
      1559.5581481733504
    ],
    [
      5878.584428035986,
      2080.474071728921
    ],
    [
      4634.699738395745,
      2830.8167008692453
    ],
    [
      4807.8655575302255,
      3415.0148494182126
    ],
    [
      5261.873696830551,
      3876.207984644642
    ],
    [
      3470.5150934969924,
      7068.249692060124
    ],
    [
      2752.8131056686925,
      6456.655677715352
    ],
    [
      2126.0230631709856,
      5096.769669026562
    ],
    [
      1233.3406321125894,
      5497.977820778063
    ]
  ],
  "target_points_display_xy": [
    [
      8797.581911426389,
      2410.540661425861
    ],
    [
      8280.157977285573,
      2023.814452595554
    ],
    [
      8396.787866089255,
      2035.7786705407962
    ],
    [
      7610.9759321546635,
      1279.0703186674727
    ],
    [
      7153.31109005224,
      1336.1885544698598
    ],
    [
      6321.17066222033,
      2441.281561224918
    ],
    [
      1559.5581481733504,
      4813.551426549105
    ],
    [
      2080.474071728921,
      5155.415571964014
    ],
    [
      2830.8167008692453,
      6399.300261604255
    ],
    [
      3415.0148494182126,
      6226.1344424697745
    ],
    [
      3876.207984644642,
      5772.126303169449
    ],
    [
      7068.249692060124,
      7563.484906503008
    ],
    [
      6456.655677715352,
      8281.186894331307
    ],
    [
      5096.769669026562,
      8907.976936829014
    ],
    [
      5497.977820778063,
      9800.65936788741
    ]
  ],
  "he_display_orientation": {
    "rotation_degrees_ccw": 90,
    "flip_horizontal": false,
    "flip_vertical": false
  },
  "linked_movement_enabled": false
}
'''

if SAVE_LANDMARKS_AND_PREVIEW:
    import json
    landmark_record = json.loads(LANDMARK_JSON)
    saved_sample = landmark_record.get('sample_folder') or landmark_record.get('sample_name')
    if not saved_sample:
        raise ValueError('Landmark JSON has no sample name.')
    alignment_result = manual_landmark_alignment.manual_landmark_align(
        root,
        saved_sample,
        output_dir=TRANSFORM_DIR,
        n_points=N_LANDMARKS,
        max_dimension=MAX_DIMENSION,
        landmarks=landmark_record,
    )
    display(alignment_result)

    precrop_tile_qc_summary = tile_pipeline.run_precrop_tile_alignment_qc(
        root,
        sample_names=[saved_sample],
        output_dir=OUTPUT_DIR,
        transform_dir=TRANSFORM_DIR,
        tile_size=1024,
        label_column=None,
        n_examples=9,
        fail_fast=True,
    )
    display(precrop_tile_qc_summary)

    preview_path = OUTPUT_DIR / saved_sample / f'{saved_sample}_precrop_alignment_tile_examples.png'
    print('Pre-cropping tile example image:', preview_path)
    if preview_path.is_file():
        display(DisplayImage(filename=str(preview_path)))
else:
    print('Paste landmark JSON and set SAVE_LANDMARKS_AND_PREVIEW=True.')


### 5c. Accept the sample or retry it


In [ ]:
ALIGNMENT_PREVIEW_OK = True
ACCEPT_SAMPLE = CURRENT_ALIGNMENT_SAMPLE

if ALIGNMENT_PREVIEW_OK:
    samples_needing_manual_alignment = manual_landmark_alignment.samples_missing_alignment(
        root,
        output_dir=TRANSFORM_DIR,
        sample_names=sample_names,
    )
    print('Accepted sample:', ACCEPT_SAMPLE)
    print('Remaining samples missing `_he_imagealignment.csv`:')
    for name in samples_needing_manual_alignment:
        print(f'  - {name}')
    if not samples_needing_manual_alignment:
        print('  (none)')
    print('NEXT_ALIGNMENT_SAMPLE =', next(iter(samples_needing_manual_alignment), None))
else:
    print('Do not move on yet. Re-run 5a/5b for this sample with improved landmarks.')
    print('Set CURRENT_ALIGNMENT_SAMPLE_OVERRIDE =', repr(ACCEPT_SAMPLE), 'in 5a if needed.')


## 6. Optional: generate broad alignment QC plots


In [ ]:
RUN_OPTIONAL_ALIGNMENT_QC = True

if RUN_OPTIONAL_ALIGNMENT_QC:
    alignment_qc_summary = tile_pipeline.run_alignment_qc(
        root,
        sample_names=sample_names,
        output_dir=OUTPUT_DIR,
        transform_dir=TRANSFORM_DIR,
        label_column=None,
        fail_fast=True,
    )
    display(alignment_qc_summary)

    alignment_failures = alignment_qc_summary.loc[alignment_qc_summary['status'] != 'success']
    if not alignment_failures.empty:
        details = alignment_failures[['sample', 'error']].to_string(index=False)
        raise RuntimeError(f'Fix alignment failures before tile creation:\n{details}')
else:
    print('Optional broad alignment QC skipped. The 9-tile pre-cropping examples are the main QC gate.')


### Optional: check broad alignment QC images


In [ ]:
if 'alignment_qc_summary' not in globals():
    print('Run the optional broad alignment QC cell first if you want to view these plots.')
else:
    alignment_qc_sample_names = alignment_qc_summary['sample'].dropna().astype(str).tolist()
    alignment_dropdown = widgets.Dropdown(
        options=alignment_qc_sample_names,
        value=alignment_qc_sample_names[0],
        description='Sample',
        layout=widgets.Layout(width='520px'),
    )
    alignment_previous = widgets.Button(description='Previous', icon='chevron-left')
    alignment_next = widgets.Button(description='Next', icon='chevron-right')
    alignment_output = widgets.Output()

    def show_alignment_sample(sample_name):
        sample_index = alignment_qc_sample_names.index(sample_name) + 1
        plot_path = OUTPUT_DIR / sample_name / f'{sample_name}_xenium_he_alignment.png'
        with alignment_output:
            clear_output(wait=True)
            print(f'Alignment QC sample {sample_index} / {len(alignment_qc_sample_names)}: {sample_name}')
            print(plot_path)
            if plot_path.is_file():
                display(DisplayImage(filename=str(plot_path)))
            else:
                print('Missing alignment image. Re-run optional broad alignment QC.')

    def move_alignment(delta):
        index = alignment_qc_sample_names.index(alignment_dropdown.value)
        alignment_dropdown.value = alignment_qc_sample_names[(index + delta) % len(alignment_qc_sample_names)]

    alignment_dropdown.observe(lambda change: show_alignment_sample(change['new']), names='value')
    alignment_previous.on_click(lambda _button: move_alignment(-1))
    alignment_next.on_click(lambda _button: move_alignment(1))
    display(widgets.HBox([alignment_previous, alignment_next, alignment_dropdown]))
    display(alignment_output)
    show_alignment_sample(alignment_dropdown.value)


## 7. Write tile input manifest


In [ ]:
TILE_INPUT_MANIFEST = OUTPUT_DIR / 'tile_input_manifest.csv'
tile_input_manifest = tile_pipeline.write_tile_input_manifest(
    root,
    sample_names=samples_for_tile_manifest,
    output_dir=OUTPUT_DIR,
    transform_dir=TRANSFORM_DIR,
    manifest_path=TILE_INPUT_MANIFEST,
)
display(tile_input_manifest)
print('Wrote tile input manifest:', TILE_INPUT_MANIFEST)


## 8. Next step
